# PLS Grid Search — I(V) / dI/dV / Both
Grid search trên cả 3 chế độ feature:
- **`I`** : dòng điện I(V) gốc
- **`dIdV`** : đạo hàm dI/dV
- **`both`** : ghép I(V) + dI/dV

Kết quả được so sánh và xếp hạng chung theo composite score.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict, LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded')

---
## ⚙️ CONFIG

In [ ]:
# ================================================================
FILE_PATH    = 'CUCOMOF.csv'

RANGE_START  = -0.2   # V bắt đầu scan
RANGE_END    =  0.8   # V kết thúc scan

WINDOW_MIN   =  0.3   # Độ rộng window nhỏ nhất (V)
WINDOW_MAX   =  0.8   # Độ rộng window lớn nhất (V)

STEP_MIN     =  5     # Số điểm tối thiểu trong window
STEP_MAX     =  20    # Số điểm tối đa trong window

WINDOW_SLIDE =  0.01  # Bước dịch window (V)

# Chế độ feature muốn chạy — bật/tắt từng mode
RUN_I        = True   # I(V)
RUN_DIDV     = True   # dI/dV
RUN_BOTH     = True   # I(V) + dI/dV

TOP_N        =  20    # Hiển thị top N models
# ================================================================

MODES = [m for m, flag in [('I', RUN_I), ('dIdV', RUN_DIDV), ('both', RUN_BOTH)] if flag]

n_ws  = len(np.arange(WINDOW_MIN, WINDOW_MAX + 0.001, 0.01))
n_pts = STEP_MAX - STEP_MIN + 1
n_st  = int((RANGE_END - RANGE_START - WINDOW_MIN) / WINDOW_SLIDE) + 1
n_combos_per_mode = n_ws * n_pts * n_st
print(f'Config: range=[{RANGE_START},{RANGE_END}]V  window=[{WINDOW_MIN},{WINDOW_MAX}]V  points=[{STEP_MIN},{STEP_MAX}]')
print(f'Modes  : {MODES}')
print(f'Combos/mode  : ~{n_combos_per_mode:,}')
print(f'Tổng combos  : ~{n_combos_per_mode * len(MODES):,}')
print(f'Ước tính thời gian: ~{n_combos_per_mode * len(MODES) * 0.005 / 60:.1f} phút')

---
## 1. Load Data

In [ ]:
CONCENTRATIONS = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
CONC_COLS = [f'{c}mM' for c in CONCENTRATIONS]
y = np.array(CONCENTRATIONS)

data_rows = []
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line or i in [0, 1, 2]:
            continue
        parts = [p.strip() for p in line.split('\t')]
        clean = [p for p in parts if p != '']
        clean = clean[:11] if len(clean) >= 11 else clean + [np.nan]*(11-len(clean))
        data_rows.append(clean)

df_raw = pd.DataFrame(data_rows, columns=['V'] + CONC_COLS)
df_raw = df_raw.apply(pd.to_numeric, errors='coerce')
df_fwd = df_raw.iloc[:df_raw['V'].idxmax()+1].copy().reset_index(drop=True)

df_scan  = df_fwd[(df_fwd['V'] >= RANGE_START - 0.005) & (df_fwd['V'] <= RANGE_END + 0.005)].copy()
V_array  = df_scan['V'].values
I_matrix = df_scan[CONC_COLS].values.astype(float)       # (n_V, 10)

# Tính dI/dV một lần cho toàn bộ
dIdV_matrix = np.gradient(I_matrix, V_array, axis=0)     # (n_V, 10)

print(f'Data: {len(df_scan)} V points  [{V_array.min():.3f}V -> {V_array.max():.3f}V]')

---
## 2. Grid Search

In [ ]:
def eval_pls_loo(X, y):
    """PLS LOO-CV, auto chọn n_components tốt nhất, giới hạn max 3."""
    scaler = StandardScaler()
    X_s    = scaler.fit_transform(X)
    loo    = LeaveOneOut()
    best_r2, best_rmse, best_nc = -999, 999, 1
    for nc in range(1, min(3, X.shape[1], len(y)-1) + 1):
        pls = PLSRegression(n_components=nc)
        yp  = cross_val_predict(pls, X_s, y, cv=loo).ravel()
        r2  = r2_score(y, yp)
        rmse = np.sqrt(mean_squared_error(y, yp))
        if r2 > best_r2:
            best_r2, best_rmse, best_nc = r2, rmse, nc
    return best_r2, best_rmse, best_nc

def composite_score(r2, rmse):
    rmse_worst = y.max() - y.min()
    rmse_norm  = np.clip(rmse / rmse_worst, 0, 1)
    return 0.6 * max(r2, 0) + 0.4 * (1 - rmse_norm)

def build_X(mode, I_win, dIdV_win, indices):
    """Tạo feature matrix theo mode."""
    if mode == 'I':
        return I_win[indices].T                                      # (10, n)
    elif mode == 'dIdV':
        return dIdV_win[indices].T                                   # (10, n)
    elif mode == 'both':
        return np.hstack([I_win[indices].T, dIdV_win[indices].T])   # (10, 2n)

# Build danh sách combinations
combos = []
for ws in np.round(np.arange(WINDOW_MIN, WINDOW_MAX + 0.001, 0.01), 3):
    for w_start in np.round(np.arange(RANGE_START, RANGE_END - ws + 0.001, WINDOW_SLIDE), 3):
        w_end = round(w_start + ws, 3)
        if w_end > RANGE_END + 0.001:
            continue
        for n_pts in range(STEP_MIN, STEP_MAX + 1):
            combos.append((w_start, w_end, ws, n_pts))

total_per_mode = len(combos)
total_all      = total_per_mode * len(MODES)
print(f'Tổng combinations: {total_per_mode:,} x {len(MODES)} modes = {total_all:,}')
print('Đang chạy grid search...\n')

all_results = []
log_every   = max(1, total_per_mode // 10)

for mode in MODES:
    print(f'--- Mode: {mode} ---')
    mode_results = []

    for i, (w_start, w_end, ws, n_pts) in enumerate(combos):
        mask  = (V_array >= w_start - 0.001) & (V_array <= w_end + 0.001)
        V_win = V_array[mask]
        I_win = I_matrix[mask]
        dIdV_win = dIdV_matrix[mask]

        if len(V_win) < 2:
            continue

        indices = np.unique(np.round(np.linspace(0, len(V_win)-1, n_pts)).astype(int))
        if len(indices) < 2:
            continue

        X = build_X(mode, I_win, dIdV_win, indices)
        if np.any(np.isnan(X)):
            continue

        r2, rmse, nc = eval_pls_loo(X, y)
        cs = composite_score(r2, rmse)

        mode_results.append({
            'mode'        : mode,
            'V_start'     : w_start,
            'V_end'       : w_end,
            'window_V'    : ws,
            'n_points'    : len(indices),
            'n_features'  : X.shape[1],
            'n_components': nc,
            'R2_LOO'      : round(r2,  5),
            'RMSE_LOO'    : round(rmse, 5),
            'composite'   : round(cs,  5),
            'V_selected'  : list(np.round(V_win[indices], 4))
        })

        if (i+1) % log_every == 0:
            pct = (i+1)/total_per_mode*100
            best_now = max(r['R2_LOO'] for r in mode_results) if mode_results else 0
            print(f'  [{mode}] {pct:5.1f}%  ({i+1:,}/{total_per_mode:,})  Best R2: {best_now:.4f}')

    all_results.extend(mode_results)
    best_mode = max(mode_results, key=lambda r: r['R2_LOO']) if mode_results else {}
    print(f'  [{mode}] Done — {len(mode_results):,} valid models  '
          f'Best R2={best_mode.get("R2_LOO","N/A")}  RMSE={best_mode.get("RMSE_LOO","N/A")}\n')

results_df = pd.DataFrame(all_results).sort_values('composite', ascending=False).reset_index(drop=True)
results_df.to_csv('pls_grid_search_v2_results.csv', index=False)
print(f'Grid search done! Tổng {len(results_df):,} valid models.')
best = results_df.iloc[0]
print(f'Best overall: mode={best["mode"]}  V=[{best["V_start"]},{best["V_end"]}]  '
      f'n={best["n_points"]}pts  R2={best["R2_LOO"]:.4f}  RMSE={best["RMSE_LOO"]:.4f}')

---
## 3. So sánh 3 Mode

In [ ]:
# Best model per mode
mode_summary = []
for mode in MODES:
    sub = results_df[results_df['mode'] == mode]
    if sub.empty: continue
    best_m = sub.iloc[0]
    mode_summary.append({
        'mode'        : mode,
        'best_R2'     : best_m['R2_LOO'],
        'best_RMSE'   : best_m['RMSE_LOO'],
        'best_V_start': best_m['V_start'],
        'best_V_end'  : best_m['V_end'],
        'best_n_pts'  : best_m['n_points'],
        'best_nComp'  : best_m['n_components'],
        'n_models'    : len(sub)
    })

summary_df = pd.DataFrame(mode_summary)
print('=== BEST PER MODE ===')
print(summary_df.to_string(index=False))

# Plot so sánh
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
mode_colors = {'I': '#3498db', 'dIdV': '#e74c3c', 'both': '#2ecc71'}

# R² best
ax = axes[0]
bars = ax.bar(summary_df['mode'], summary_df['best_R2'],
              color=[mode_colors.get(m,'gray') for m in summary_df['mode']],
              alpha=0.85, edgecolor='black')
for bar, v in zip(bars, summary_df['best_R2']):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.005, f'{v:.4f}',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Best R² (LOO-CV)', fontsize=11)
ax.set_title('Best R² per Mode', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.1); ax.grid(True, alpha=0.3, axis='y')

# RMSE best
ax = axes[1]
bars = ax.bar(summary_df['mode'], summary_df['best_RMSE'],
              color=[mode_colors.get(m,'gray') for m in summary_df['mode']],
              alpha=0.85, edgecolor='black')
for bar, v in zip(bars, summary_df['best_RMSE']):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.003, f'{v:.4f}',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Best RMSE (mM)', fontsize=11)
ax.set_title('Best RMSE per Mode', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# R² distribution per mode
ax = axes[2]
for mode in MODES:
    sub = results_df[results_df['mode'] == mode]['R2_LOO']
    ax.hist(sub, bins=40, alpha=0.5, color=mode_colors.get(mode,'gray'),
            label=mode, edgecolor='none')
ax.set_xlabel('R² (LOO-CV)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('R² Distribution per Mode', fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

plt.suptitle('Mode Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('grid_mode_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Top Models & Heatmaps

In [ ]:
cols = ['mode','V_start','V_end','window_V','n_points','n_features','n_components','R2_LOO','RMSE_LOO','composite']
print(f'=== TOP {TOP_N} MODELS (overall) ===')
print(results_df[cols].head(TOP_N).to_string(index=True))

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, hspace=0.45, wspace=0.35)

# Heatmap per mode
for mi, mode in enumerate(MODES[:3]):
    ax = fig.add_subplot(gs[0, mi])
    sub = results_df[results_df['mode'] == mode]
    if sub.empty:
        ax.set_visible(False); continue
    pivot = sub.groupby(['V_start','window_V'])['R2_LOO'].max().reset_index()
    pt    = pivot.pivot(index='window_V', columns='V_start', values='R2_LOO')
    vmin  = sub['R2_LOO'].quantile(0.1)
    vmax  = sub['R2_LOO'].max()
    im = ax.imshow(pt.values, aspect='auto', origin='lower', cmap='RdYlGn',
                   vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label='R²')
    xi = np.linspace(0, pt.shape[1]-1, 5).astype(int)
    yi = np.linspace(0, pt.shape[0]-1, 5).astype(int)
    ax.set_xticks(xi); ax.set_xticklabels([f'{v:.2f}' for v in pt.columns[xi]], fontsize=7)
    ax.set_yticks(yi); ax.set_yticklabels([f'{v:.2f}' for v in pt.index[yi]],  fontsize=7)
    ax.set_xlabel('V_start (V)', fontsize=9)
    ax.set_ylabel('Window (V)', fontsize=9)
    best_m = sub.iloc[0]
    ax.set_title(f'[{mode}]  Best R²={best_m["R2_LOO"]:.4f}\n'
                 f'V=[{best_m["V_start"]},{best_m["V_end"]}]  {best_m["n_points"]}pts',
                 fontsize=9, fontweight='bold')

# R² vs n_points per mode
ax_pts = fig.add_subplot(gs[1, 0])
for mode in MODES:
    sub = results_df[results_df['mode'] == mode]
    g   = sub.groupby('n_points')['R2_LOO'].max()
    ax_pts.plot(g.index, g.values, 'o-', lw=1.8, ms=5,
                color=mode_colors.get(mode,'gray'), label=mode)
ax_pts.set_xlabel('n_points', fontsize=10); ax_pts.set_ylabel('Max R²', fontsize=10)
ax_pts.set_title('R² vs n_points (per mode)', fontsize=11, fontweight='bold')
ax_pts.legend(fontsize=9); ax_pts.grid(True, alpha=0.3)

# R² vs window_size per mode
ax_ws = fig.add_subplot(gs[1, 1])
for mode in MODES:
    sub = results_df[results_df['mode'] == mode]
    g   = sub.groupby('window_V')['R2_LOO'].max()
    ax_ws.plot(g.index, g.values, '-', lw=1.5,
               color=mode_colors.get(mode,'gray'), label=mode)
ax_ws.set_xlabel('Window size (V)', fontsize=10); ax_ws.set_ylabel('Max R²', fontsize=10)
ax_ws.set_title('R² vs Window size (per mode)', fontsize=11, fontweight='bold')
ax_ws.legend(fontsize=9); ax_ws.grid(True, alpha=0.3)

# Top 20 composite
ax_top = fig.add_subplot(gs[1, 2])
top20  = results_df.head(20)
bar_c  = [mode_colors.get(m,'gray') for m in top20['mode']]
labels = [f'[{m}] [{r.V_start:.2f},{r.V_end:.2f}] {r.n_points}pts'
          for m, (_, r) in zip(top20['mode'], top20.iterrows())]
ax_top.barh(range(len(top20)), top20['composite'], color=bar_c, alpha=0.85, edgecolor='black', lw=0.4)
ax_top.set_yticks(range(len(top20)))
ax_top.set_yticklabels(labels, fontsize=6.5)
ax_top.invert_yaxis()
ax_top.set_xlabel('Composite Score', fontsize=10)
ax_top.set_title(f'Top {TOP_N} Overall', fontsize=11, fontweight='bold')
for m, patch in zip(top20['mode'], ax_top.patches):
    pass  # colors already set
# Legend
from matplotlib.patches import Patch
ax_top.legend(handles=[Patch(color=mode_colors.get(m,'gray'), label=m) for m in MODES],
              fontsize=8, loc='lower right')
ax_top.grid(True, alpha=0.3, axis='x')

# R² distribution overlay
ax_dist = fig.add_subplot(gs[2, :])
for mode in MODES:
    sub = results_df[results_df['mode'] == mode]['R2_LOO']
    ax_dist.hist(sub, bins=60, alpha=0.45, color=mode_colors.get(mode,'gray'),
                 label=f'{mode} (best={sub.max():.4f}, median={sub.median():.4f})', edgecolor='none')
ax_dist.set_xlabel('R² (LOO-CV)', fontsize=11); ax_dist.set_ylabel('Count', fontsize=11)
ax_dist.set_title('R² Distribution — All Modes', fontsize=12, fontweight='bold')
ax_dist.legend(fontsize=9); ax_dist.grid(True, alpha=0.3)

plt.savefig('grid_overview_v2.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Best Model per Mode — Chi tiết

In [ ]:
def rebuild_and_plot(mode, row, ax_cv, ax_pred, ax_res):
    """Rebuild model và plot 3 subplot cho 1 mode."""
    V_best = np.array(row['V_selected'])

    # Lấy I và dI/dV tại các V đã chọn
    I_rows, dIdV_rows = [], []
    for tv in V_best:
        idx = np.argmin(np.abs(V_array - tv))
        I_rows.append(I_matrix[idx])
        dIdV_rows.append(dIdV_matrix[idx])
    I_sel    = np.array(I_rows).T       # (10, n)
    dIdV_sel = np.array(dIdV_rows).T    # (10, n)

    if mode == 'I':
        X = I_sel
    elif mode == 'dIdV':
        X = dIdV_sel
    elif mode == 'both':
        X = np.hstack([I_sel, dIdV_sel])

    scaler  = StandardScaler()
    X_s     = scaler.fit_transform(X)
    pls     = PLSRegression(n_components=int(row['n_components']))
    pls.fit(X_s, y)
    y_loo   = cross_val_predict(pls, X_s, y, cv=LeaveOneOut()).ravel()

    lc = plt.cm.plasma(np.linspace(0.1, 0.9, len(CONCENTRATIONS)))

    # CV curves
    for i, (col, conc) in enumerate(zip(CONC_COLS, CONCENTRATIONS)):
        ax_cv.plot(df_fwd['V'], df_fwd[col], color=lc[i], lw=1.2, alpha=0.8)
    for v in V_best:
        ax_cv.axvline(x=v, color='red', lw=0.9, alpha=0.7, linestyle='--')
    ax_cv.axvspan(row['V_start'], row['V_end'], alpha=0.08, color='red')
    ax_cv.set_xlabel('V (V)', fontsize=9); ax_cv.set_ylabel('I (µA)', fontsize=9)
    ax_cv.set_title(f'[{mode}] CV  [{row["V_start"]},{row["V_end"]}]V  {row["n_points"]}pts',
                    fontsize=9, fontweight='bold')
    ax_cv.grid(True, alpha=0.3)

    # Predicted vs Actual
    ax_pred.scatter(y, y_loo, s=80, color=mode_colors.get(mode,'gray'), zorder=5)
    lim = [y.min()-0.4, y.max()+0.4]
    ax_pred.plot(lim, lim, 'r--', lw=1.5)
    for a, p in zip(y, y_loo):
        ax_pred.annotate(f'{a}', (a, p), textcoords='offset points', xytext=(4,3), fontsize=7)
    ax_pred.set_xlabel('Actual', fontsize=9); ax_pred.set_ylabel('Predicted', fontsize=9)
    ax_pred.set_title(f'[{mode}] R²={row["R2_LOO"]:.4f}  RMSE={row["RMSE_LOO"]:.4f}mM',
                      fontsize=9, fontweight='bold')
    ax_pred.grid(True, alpha=0.3)

    # Residuals
    res = y_loo - y
    ax_res.bar(range(len(y)), res, color=['#e74c3c' if r<0 else '#3498db' for r in res],
               alpha=0.85, edgecolor='black')
    ax_res.axhline(0, color='black', lw=1)
    ax_res.set_xticks(range(len(y)))
    ax_res.set_xticklabels([str(v) for v in y], fontsize=7)
    for i, ri in enumerate(res):
        ax_res.annotate(f'{ri:.2f}', (i, ri), textcoords='offset points',
                        xytext=(0, 4 if ri>=0 else -11), ha='center', fontsize=7)
    ax_res.set_xlabel('Sample', fontsize=9); ax_res.set_ylabel('Residual', fontsize=9)
    ax_res.set_title(f'[{mode}] Residuals', fontsize=9, fontweight='bold')
    ax_res.grid(True, alpha=0.3, axis='y')

n_modes = len(MODES)
fig, axes = plt.subplots(n_modes, 3, figsize=(15, 5*n_modes))
if n_modes == 1:
    axes = [axes]

for mi, mode in enumerate(MODES):
    sub = results_df[results_df['mode'] == mode]
    if sub.empty: continue
    best_row = sub.iloc[0]
    print(f'[{mode}] Best: V=[{best_row["V_start"]},{best_row["V_end"]}]  '
          f'n={best_row["n_points"]}pts  nComp={best_row["n_components"]}  '
          f'R2={best_row["R2_LOO"]:.4f}  RMSE={best_row["RMSE_LOO"]:.4f}')
    rebuild_and_plot(mode, best_row, axes[mi][0], axes[mi][1], axes[mi][2])

plt.suptitle('Best Model per Mode', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('grid_best_per_mode.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Best Overall Model

In [ ]:
best = results_df.iloc[0]
print('=== BEST OVERALL MODEL ===')
for k in ['mode','V_start','V_end','window_V','n_points','n_features','n_components','R2_LOO','RMSE_LOO','composite']:
    print(f'  {k:<15}: {best[k]}')
print(f'  {"V_selected":<15}: {best["V_selected"]}')

# Rebuild best overall
V_best   = np.array(best['V_selected'])
I_rows, dIdV_rows = [], []
for tv in V_best:
    idx = np.argmin(np.abs(V_array - tv))
    I_rows.append(I_matrix[idx])
    dIdV_rows.append(dIdV_matrix[idx])
I_sel    = np.array(I_rows).T
dIdV_sel = np.array(dIdV_rows).T

if best['mode'] == 'I':
    X_best = I_sel
elif best['mode'] == 'dIdV':
    X_best = dIdV_sel
elif best['mode'] == 'both':
    X_best = np.hstack([I_sel, dIdV_sel])

scaler   = StandardScaler()
X_best_s = scaler.fit_transform(X_best)
pls_best = PLSRegression(n_components=int(best['n_components']))
pls_best.fit(X_best_s, y)
y_loo    = cross_val_predict(pls_best, X_best_s, y, cv=LeaveOneOut()).ravel()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
lc = plt.cm.plasma(np.linspace(0.1, 0.9, len(CONCENTRATIONS)))

# CV curves
ax1 = axes[0]
for i, (col, conc) in enumerate(zip(CONC_COLS, CONCENTRATIONS)):
    ax1.plot(df_fwd['V'], df_fwd[col], color=lc[i], label=f'{conc}mM', lw=1.3, alpha=0.8)
for v in V_best:
    ax1.axvline(x=v, color='red', lw=1.0, alpha=0.7, linestyle='--')
ax1.axvspan(best['V_start'], best['V_end'], alpha=0.08, color='red')
ax1.set_xlabel('Potential (V)', fontsize=10); ax1.set_ylabel('Current (µA)', fontsize=10)
ax1.set_title(f'CV + Selected V  [{best["V_start"]},{best["V_end"]}]V\nmode={best["mode"]}  {best["n_points"]}pts',
              fontweight='bold')
ax1.legend(fontsize=6, ncol=2); ax1.grid(True, alpha=0.3)

# Predicted vs Actual
ax2 = axes[1]
ax2.scatter(y, y_loo, s=100, color=mode_colors.get(best['mode'],'steelblue'), zorder=5)
lim = [y.min()-0.4, y.max()+0.4]
ax2.plot(lim, lim, 'r--', lw=1.5)
for a, p in zip(y, y_loo):
    ax2.annotate(f'{a}mM', (a, p), textcoords='offset points', xytext=(5,4), fontsize=8)
ax2.set_xlabel('Actual (mM)', fontsize=10); ax2.set_ylabel('Predicted (mM)', fontsize=10)
ax2.set_title(f'Predicted vs Actual (LOO)\nR²={best["R2_LOO"]:.4f}  RMSE={best["RMSE_LOO"]:.4f}mM',
              fontweight='bold')
ax2.grid(True, alpha=0.3)

# Residuals
ax3 = axes[2]
res = y_loo - y
ax3.bar(y, res, width=0.08, color=['#e74c3c' if r<0 else '#3498db' for r in res],
        alpha=0.85, edgecolor='black')
ax3.axhline(0, color='black', lw=1)
for yi, ri in zip(y, res):
    ax3.annotate(f'{ri:.2f}', (yi, ri), textcoords='offset points',
                 xytext=(0, 5 if ri>=0 else -13), ha='center', fontsize=8)
ax3.set_xlabel('Actual (mM)', fontsize=10); ax3.set_ylabel('Residual (mM)', fontsize=10)
ax3.set_title('Residuals (LOO-CV)', fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Best Overall  mode={best["mode"]}  V=[{best["V_start"]},{best["V_end"]}]  '
             f'{best["n_points"]}pts  nComp={best["n_components"]}  '
             f'R²={best["R2_LOO"]:.4f}  RMSE={best["RMSE_LOO"]:.4f}mM',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('grid_best_overall.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Prediction Table ===')
df_res = pd.DataFrame({
    'Actual(mM)'      : y,
    'Predicted_LOO'   : y_loo.round(4),
    'Residual'        : (y_loo-y).round(4),
    'AbsError'        : np.abs(y_loo-y).round(4)
})
print(df_res.to_string(index=False))
print(f'\nMean Abs Error: {np.abs(y_loo-y).mean():.4f} mM')